# NYC Housing and Street Safety: Master Notebook

**Course:** Data Engineering and Visualization, GIU, Winter 2026 (Milestone 1)

**Datasets:**
- H5: DOB NOW: Build Job Application Filings, https://data.cityofnewyork.us/d/w9ak-ipjd
- T4: Serious Injury Response, Tracking and Analysis Program (SIRTA), https://data.cityofnewyork.us/d/xeqp-qz8h

**Contract:** file names, column names, and definitions are in `DATA_CONTRACT.md`.

> **How this notebook is built.** Each member works in their own notebook with these same section headings. Member 6 assembles the final version by pasting each member's cells under the matching heading. Rules: relative paths only, set random seeds, write a markdown justification for every cleaning or analysis decision, comment complex code, and make the notebook run top to bottom.

## 0. Setup

In [1]:
import sys
from pathlib import Path

# Make the repo root importable so `from src...` works when the notebook runs from notebooks/
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from src.load_data import load_housing, load_crashes, CLEAN_DIR

SEED = 42
np.random.seed(SEED)
pd.set_option("display.max_columns", 50)

## 1. Project overview: the question and why it matters to New Yorkers

*Owner: M6. Write 2 to 3 short paragraphs after the findings are known.*

- What did we want to know?
- Why does it matter for NYC residents, planners, and street-safety teams?
- One-paragraph summary of the approach.

## 2. Datasets: overview, links, and reasoning for the selection

*Owners: M1 (H5) and M2 (SIRTA).*

For each dataset report: link and dataset ID, number of rows and columns, relevant geographic and temporal attributes, known issues, and why it was chosen. Explain why the two can be integrated meaningfully, and how the difference in size and granularity is handled (see section 6). Justify the filtered download of H5 (date window and selected columns) because the full dataset is too large.

In [ ]:
# Raw downloads are cached in data/raw/ (git-ignored). Use refresh=True to download again.
df_housing_raw = load_housing()
df_crash_raw = load_crashes()

print("H5 housing :", df_housing_raw.shape)
print("SIRTA crash:", df_crash_raw.shape)

## 3. EDA: housing (H5)

*Owner: M1.* Descriptive statistics, distributions of key columns (job type, filing year, proposed units, cost), missing-value overview, and initial plots. State what each plot shows and what problem or pattern it reveals.

In [ ]:
# TODO (M1): housing EDA

## 4. EDA: crashes (SIRTA)

*Owner: M2.* Descriptive statistics, crashes per year and month, severity distribution, missing-value overview (especially ZIP, latitude, longitude), and initial plots. State what each plot shows.

In [ ]:
# TODO (M2): crash EDA

## 5. Pre-integration cleaning

For every step: what the problem is, how it was detected, what was done, and why (drop vs impute, alternatives considered). Show before and after plots or counts to validate each step.

### 5.1 Housing (M1)

Missing values, outliers (IQR and placeholder codes such as 998/999 and 9999999), inconsistent values, standardized formats (ZIP from `postcode`, borough, dates), duplicates (job filing number rule), derived columns. Output: `data/clean/housing_clean.csv` and `data/clean/housing_sites_points.csv`.

In [ ]:
# TODO (M1): housing cleaning

### 5.2 Crashes (M2)

Null IDs and dates, duplicates, time format, ZIP and borough recovery by spatial join (record `zip_source` and `borough_source`), invalid coordinates, derived flags. Output: `data/clean/crash_clean.csv`.

In [ ]:
# TODO (M2): crash cleaning

## 6. Integration

*Owner: M3.* Explain the join keys and the two grains (ZIP by year, borough by month), why those levels fit the research questions, why a row-level join is not used, how the data was aggregated, and the alternatives considered (for example ZIP by month, which is too sparse for SIRTA). Output: `zip_year_integrated.csv` and `borough_month_integrated.csv`.

In [ ]:
# TODO (M3): integration

## 7. Post-integration cleaning

*Owner: M3.* New missing values from the join (and why zero-fill is right for counts), redundant or inconsistent columns, data type mismatches, final assertions (no duplicate or null keys, exact contract column names), and the coverage report.

In [ ]:
# TODO (M3): post-integration cleaning and checks

## 8. Research questions and visualizations

Each question gets: the question, why it matters, the chart (as a function from `src/charts/`), how to read it, and the caveats. All findings are associations, never causation.

### Q1 (M1)

**Question:** Do ZIPs with a surge in new-building filings see a rise in serious crashes the following year, compared with ZIPs without a surge?

**Planned chart:** zip by year; scatter of change in new-building filings vs next-year change in crashes

**Why this chart answers the question:** TODO

**Result and interpretation:** TODO

**Caveats:** TODO

In [ ]:
# TODO (M1): Q1 chart

### Q2 (M1)

**Question:** Are serious crashes closer to new-building sites than to demolition sites?

**Planned chart:** row-level points; grouped bars of crash share by distance band and site type

**Why this chart answers the question:** TODO

**Result and interpretation:** TODO

**Caveats:** TODO

In [ ]:
# TODO (M1): Q2 chart

### Q3 (M2)

**Question:** Are crashes in construction-heavy ZIPs more likely to be fatal than in quiet ZIPs?

**Planned chart:** zip by year; box plot of fatal-crash share by construction tercile

**Why this chart answers the question:** TODO

**Result and interpretation:** TODO

**Caveats:** TODO

In [ ]:
# TODO (M2): Q3 chart

### Q4 (M2)

**Question:** Do construction-heavy ZIPs have a higher share of crashes at repeat-danger locations?

**Planned chart:** zip by year; bar or violin by construction tercile

**Why this chart answers the question:** TODO

**Result and interpretation:** TODO

**Caveats:** TODO

In [ ]:
# TODO (M2): Q4 chart

### Q5 (M3)

**Question:** Do housing filings and SIRTA transportation safety records show similar monthly patterns across NYC boroughs?

**Planned chart:** Monthly line charts comparing Housing filings and SIRTA serious crash counts for each borough, accompanied by a lag-correlation chart examining shifts from three months before to three months after.

**Why this chart answers the question:** Q5 examines whether housing filing activity and serious transportation crashes follow similar patterns over time. The first chart shows both datasets month by month for each NYC borough, making it possible to identify increases, decreases, seasonal patterns and periods when the two series move together or differently. Because the number of housing filings and crashes differs substantially, each series is normalized to its own average within each borough for visual comparison. The second chart calculates correlations across lags of -3 to +3 months to examine whether changes in one series tend to appear earlier or later than changes in the other. Together, the charts help identify possible temporal associations rather than relying only on annual totals.

**Result and interpretation:** Housing filings and serious crashes do not show a strong, consistent monthly relationship across the five boroughs. Same-month correlations are weak in the Bronx (0.154), Brooklyn (0.079), Manhattan (0.086) and Queens (0.024), while Staten Island shows a weak negative relationship (-0.222). The strongest positive lagged relationships occur when housing filings lead crash counts by two months in Brooklyn (0.340) and the Bronx (0.265). Manhattan (-0.243) and Queens (-0.252) show their strongest absolute correlations at a lag of -3 months, while Staten Island (-0.306) is strongest at -2 months. These correlations are still relatively small and do not follow one consistent direction across boroughs. Overall, the charts suggest that monthly housing filings are not a reliable indicator of when serious crashes will increase, although some borough-specific patterns are worth exploring further.

**Caveats:** Housing filings represent submitted paperwork rather than confirmed construction activity, while the SIRTA dataset contains serious transportation incidents rather than all NYC traffic crashes. The analysis uses borough-level monthly totals, which can hide differences between neighborhoods and ZIP codes. Changes in reporting, seasonal travel patterns, borough size and exposure to traffic may influence the trends. Normalization makes the patterns easier to compare but does not make their absolute numbers comparable. Lag correlations may also reflect common trends, seasonality or autocorrelation rather than a meaningful relationship. The monthly dataset covers January 2022 through June 2026, including a partial final year, so comparisons should account for differences in time coverage. This is exploratory analysis, not evidence of causation.

In [5]:

from pathlib import Path
import pandas as pd
import numpy as np
import plotly.express as px

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent

monthly = pd.read_csv(
    ROOT / "data/clean/borough_month_integrated.csv"
)

monthly["date"] = pd.to_datetime(
    monthly["year_month"].astype(str).str[:7] + "-01"
)

monthly = monthly.sort_values(["borough", "date"]).copy()

# Normalize each series to its average within each borough.
# 100 means equal to that borough's monthly average.
monthly["housing_index"] = monthly.groupby("borough")["h_filings_total"].transform(
    lambda x: 100 * x / x.mean() if x.mean() > 0 else np.nan
)
monthly["sirta_index"] = monthly.groupby("borough")["t_crashes"].transform(
    lambda x: 100 * x / x.mean() if x.mean() > 0 else np.nan
)

chart_data = monthly.melt(
    id_vars=["borough", "date"],
    value_vars=["housing_index", "sirta_index"],
    var_name="series",
    value_name="index"
)

chart_data["series"] = chart_data["series"].map({
    "housing_index": "Housing filings",
    "sirta_index": "SIRTA serious crashes"
})

fig_q5 = px.line(
    chart_data,
    x="date",
    y="index",
    color="series",
    facet_row="borough",
    title="Q5: Monthly Housing and SIRTA Patterns by Borough",
    labels={"date": "Month", "index": "Index (borough average = 100)"},
    height=1100
)

fig_q5.update_yaxes(matches=None)
fig_q5.update_layout(hovermode="x unified")
fig_q5.show()

# Correlation at different monthly lags, using original counts
lag_results = []

for borough, group in monthly.groupby("borough"):
    group = group.sort_values("date").set_index("date").asfreq("MS")

    for lag in range(-3, 4):
        paired = pd.DataFrame({
            "housing": group["h_filings_total"].shift(lag),
            "sirta": group["t_crashes"]
        }).dropna()

        if (
            len(paired) >= 6
            and paired["housing"].nunique() > 1
            and paired["sirta"].nunique() > 1
        ):
            correlation = paired["housing"].corr(paired["sirta"])
        else:
            correlation = np.nan

        lag_results.append({
            "borough": borough,
            "lag_months": lag,
            "correlation": correlation,
            "observations": len(paired)
        })

lag_df = pd.DataFrame(lag_results)

fig_q5_lag = px.line(
    lag_df,
    x="lag_months",
    y="correlation",
    color="borough",
    markers=True,
    hover_data=["observations"],
    title="Q5: Housing–SIRTA Monthly Lag Correlations",
    labels={
        "lag_months": "Housing lag in months",
        "correlation": "Pearson correlation"
    }
)

fig_q5_lag.add_hline(y=0, line_dash="dash")
fig_q5_lag.show()

print("Q5: Same-month correlation by borough")
display(
    lag_df.loc[lag_df["lag_months"] == 0,
               ["borough", "correlation", "observations"]]
    .round(3)
)

print("Q5 — Strongest absolute correlation by borough")
strongest = (
    lag_df.dropna(subset=["correlation"])
    .assign(abs_corr=lambda d: d["correlation"].abs())
    .sort_values("abs_corr", ascending=False)
    .drop_duplicates("borough")
)

display(
    strongest[
        ["borough", "lag_months", "correlation", "observations"]
    ].sort_values("borough").round(3)
)


Q5: Same-month correlation by borough


,borough,correlation,observations
3,Bronx,0.154,54
10,Brooklyn,0.079,54
17,Manhattan,0.086,54
24,Queens,0.024,54
31,Staten Island,-0.222,54


Q5 — Strongest absolute correlation by borough


,borough,lag_months,correlation,observations
5,Bronx,2,0.265,52
12,Brooklyn,2,0.340,52
14,Manhattan,-3,-0.243,51
21,Queens,-3,-0.252,51
29,Staten Island,-2,-0.306,52


### Q6 (M3)
**Question:** Which ZIP codes have more or fewer SIRTA transportation safety records than expected based on their housing filing activity?

**Planned chart:** ZIP-level scatter plot comparing average monthly housing filings and serious crash counts, with a linear regression trendline and highlighted ZIP codes showing the largest positive and negative residuals.

**Why this chart answers the question:** Q6 investigates whether ZIP codes with more housing filing activity also tend to have more serious transportation crashes. Each point represents one ZIP code, with its total filings and serious crash records aggregated across the complete years 2022–2025 and converted into monthly averages. A regression line provides an estimated crash count based on housing activity. The vertical distance between each point and the line is its residual. ZIP codes above the line have more serious crash records than the fitted model estimates, while those below have fewer. Highlighting the five largest positive and negative residuals makes it easier to identify ZIP codes that differ most from the overall pattern.

**Result and interpretation:** Across 226 ZIP codes, housing filing activity and serious crash records show a moderate positive association (Pearson correlation = 0.538, R² = 0.290). The fitted regression equation is approximately monthly serious crashes = 0.0146 × monthly housing filings + 0.5293. This means housing filing activity explains approximately 29% of the variation in average monthly serious crash counts across ZIP codes in this simple linear model, leaving most variation unexplained. ZIP code 11236 in Brooklyn has the largest positive residual, recording approximately 3.646 serious crashes per month compared with 1.054 estimated by the model, a difference of +2.592 crashes per month. Other ZIP codes with higher-than-estimated records include 11234, 11207, 10456 and 11212. By contrast, ZIP code 10022 in Manhattan records approximately 1.604 serious crashes per month compared with 3.106 estimated, a residual of -1.502. Other lower-than-estimated ZIP codes include 10014, 10011, 10003 and 10013. These findings suggest that filing activity is associated with crash counts, but additional factors are necessary to understand why some ZIP codes have more or fewer crashes than estimated.

**Caveats:** This analysis is descriptive and does not establish causation. Housing filings are administrative records rather than direct measurements of construction activity, and SIRTA records represent a specific subset of serious transportation crashes. ZIP codes differ in population, street length, traffic exposure and geographic size, but the regression does not adjust for these factors. Some crash records cannot be assigned to ZIP codes, and spatially recovered ZIP codes have geographic uncertainty. The analysis combines the four complete years from 2022 to 2025, so it does not capture differences in timing within those years. A simple linear model may not adequately describe the relationship, particularly for ZIP codes with unusually high counts. Residuals identify differences from the fitted line, not statistically validated safety outcomes.

In [3]:

from pathlib import Path
import pandas as pd
import numpy as np
import plotly.express as px

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent

zip_year = pd.read_csv(
    ROOT / "data/clean/zip_year_integrated.csv",
    dtype={"zip": str}
)

# Use the complete years 2022–2025.
data_q6 = zip_year[
    zip_year["year"].between(2022, 2025)
].copy()

# One point per ZIP code, combining all four years.
zip_analysis = (
    data_q6.groupby("zip", as_index=False)
    .agg(
        housing_filings=("h_filings_total", "sum"),
        sirta_crashes=("t_crashes", "sum"),
        borough=("borough", "first")
    )
)

# Four years = 48 months
zip_analysis["filings_per_month"] = zip_analysis["housing_filings"] / 48
zip_analysis["sirta_per_month"] = zip_analysis["sirta_crashes"] / 48

x = zip_analysis["filings_per_month"].to_numpy(dtype=float)
y = zip_analysis["sirta_per_month"].to_numpy(dtype=float)

# Fit the linear regression model
slope, intercept = np.polyfit(x, y, 1)

zip_analysis["predicted_sirta"] = slope * x + intercept

# Residual = actual - predicted
zip_analysis["residual"] = (
    zip_analysis["sirta_per_month"]
    - zip_analysis["predicted_sirta"]
)

# Highlight the five highest and five lowest residuals
zip_analysis["category"] = "Other ZIP codes"

high_idx = zip_analysis.nlargest(5, "residual").index
low_idx = zip_analysis.nsmallest(5, "residual").index

zip_analysis.loc[high_idx, "category"] = "Higher than estimated"
zip_analysis.loc[low_idx, "category"] = "Lower than estimated"

fig_q6 = px.scatter(
    zip_analysis,
    x="filings_per_month",
    y="sirta_per_month",
    color="category",
    hover_name="zip",
    hover_data=[
        "borough",
        "housing_filings",
        "sirta_crashes",
        "predicted_sirta",
        "residual"
    ],
    title="Q6: Housing Activity and Serious Crash Records by ZIP",
    labels={
        "filings_per_month": "Average monthly housing filings",
        "sirta_per_month": "Average monthly SIRTA crashes",
        "category": "Residual group"
    }
)

# Draw fitted regression line
line_x = np.linspace(x.min(), x.max(), 100)

fig_q6.add_scatter(
    x=line_x,
    y=slope * line_x + intercept,
    mode="lines",
    name="Linear regression",
    line=dict(color="black", dash="dash")
)

fig_q6.update_layout(height=650)
fig_q6.show()

# Numerical summaries for the notebook
correlation = zip_analysis[
    ["filings_per_month", "sirta_per_month"]
].corr().iloc[0, 1]

r_squared = correlation ** 2

print("Q6 Results")
print("ZIP codes analyzed:", len(zip_analysis))
print("Pearson correlation:", round(correlation, 3))
print("R-squared:", round(r_squared, 3))
print("Regression slope:", round(slope, 4))
print("Regression intercept:", round(intercept, 4))

print("\nFive ZIP codes with more crashes than estimated:")
display(
    zip_analysis.nlargest(5, "residual")[
        ["zip", "borough", "filings_per_month",
         "sirta_per_month", "predicted_sirta", "residual"]
    ].round(3)
)

print("\nFive ZIP codes with fewer crashes than estimated:")
display(
    zip_analysis.nsmallest(5, "residual")[
        ["zip", "borough", "filings_per_month",
         "sirta_per_month", "predicted_sirta", "residual"]
    ].round(3)
)


Q6 Results
ZIP codes analyzed: 226
Pearson correlation: 0.538
R-squared: 0.29
Regression slope: 0.0146
Regression intercept: 0.5293

Five ZIP codes with more crashes than estimated:


,zip,borough,filings_per_month,sirta_per_month,predicted_sirta,residual
165,11236,Brooklyn,35.854,3.646,1.054,2.592
163,11234,Brooklyn,40.771,3.188,1.126,2.061
137,11207,Brooklyn,61.000,3.250,1.423,1.827
98,10456,Bronx,38.042,2.854,1.086,1.768
142,11212,Brooklyn,51.708,3.042,1.286,1.755



Five ZIP codes with fewer crashes than estimated:


,zip,borough,filings_per_month,sirta_per_month,predicted_sirta,residual
19,10022,Manhattan,176.000,1.604,3.106,-1.502
12,10014,Manhattan,107.354,0.625,2.101,-1.476
9,10011,Manhattan,136.812,1.125,2.533,-1.408
2,10003,Manhattan,130.542,1.271,2.441,-1.170
11,10013,Manhattan,118.021,1.125,2.257,-1.132


### Q7 (M4)

**Question:** Is the link between construction and serious crashes stable from 2022 to 2025, or does it appear only in some years and boroughs?

**Planned chart:** zip by year; heatmap of Spearman correlation, borough by year

**Why this chart answers the question:** The heatmap computes a Spearman rank correlation between new-building filings and street crashes across ZIP codes, separately for each borough and year. Each cell is one borough-year, so reading across a row shows whether the relationship holds, flips or fades over time. Spearman is used because filing and crash counts are skewed and the question is about ranking ZIPs, not exact amounts. A borough-year with fewer than 5 ZIPs is left blank, because a correlation on so few points is too noisy. The caption compares the mean rho with and without 2025, which tests whether the conclusion depends on the latest year. The second chart shows where each crash's ZIP came from, so readers can judge how far to trust each year.


**Result and interpretation:** The correlation is positive in almost every borough and year. Manhattan is strongest (0.58 to 0.83) and Brooklyn weakest (0.17 to 0.40). The mean rho is 0.47 with all years and 0.48 without 2025, so the latest year does not change the picture. The one exception is Staten Island in 2025 (-0.21), which rests on very few ZIPs and is probably noise. Overall the link looks stable across 2022 to 2025, but it is an association between busy areas, not evidence of cause.

**Caveats:** Association, not causation: ZIPs with more filings may simply be busier. ZIP coverage differs by year: only 16.3% of 2022 crashes had an original ZIP (73.2% were recovered by spatial join), against 80.3% in 2025, so part of any year-to-year change may reflect how ZIPs were filled in. 8.9% of the 13,309 crashes have no ZIP and are excluded, and this hits fatal crashes hardest (33.8% missing versus 6.7% for non-fatal) and highway crashes (21.5%). 44 crashes carry the unverified ZIP 10000. 2026 is a partial year and is not in the ZIP-year table. Boroughs with few ZIPs, especially Staten Island, give unstable cells.

In [ ]:
from pathlib import Path
import pandas as pd
from src.charts import q07, q08

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
zip_year = pd.read_csv(ROOT / "data/clean/zip_year_integrated.csv", dtype={"zip": str})

q07.make_fig(zip_year).show()
q07.make_coverage_fig(ROOT / "data/clean/coverage_report.csv").show()

: 

### Q8 (M4)

**Question:** Do street-facing filings (sidewalk sheds, curb cuts) track serious crashes better than other alterations?

**Planned chart:** zip by year; grouped bars of correlation per metric, by borough

**Why this chart answers the question:** The grouped bars compare three filing types on the same crash measure: sidewalk sheds and curb cuts (street-facing) against alteration certificates of occupancy (other alterations, the comparison group). For each, the chart shows the Spearman rho across ZIP-years, for all boroughs combined and for each borough, on one axis from -1 to 1. If street-facing filings track crashes better, their bars should be consistently higher than the alteration bar. The zero line marks no relationship. Groups with fewer than 5 ZIP-years are skipped.


**Result and interpretation:** Sidewalk sheds track street crashes about as closely as other alterations (rho around 0.5 to 0.9 in most boroughs), and curb cuts track them less closely in every borough. So the data does not support the idea that street-facing filings track serious crashes better than other alterations: only sidewalk sheds match the comparison group, and curb cuts fall below it. Brooklyn is weak for all three types (about 0.15 to 0.3).

**Caveats:** Association, not causation: sheds and curb cuts cluster in busy, dense areas. The rho is computed on ZIP-years pooled across 2022 to 2025, so it mixes differences between ZIPs and between years. Crashes without a ZIP are excluded, and fatal and highway crashes are the most affected. The chart has no confidence intervals, so small differences between bars should not be read as real.

In [ ]:
q08.make_fig(zip_year).show()

### Q9 (M5)

**Question:** Where do high-construction and high-injury ZIPs overlap spatially?

**Planned chart:** a bivariate choropleth. Each ZIP is ranked into a low, medium, or high third for new-building filings and for severe injuries (summed over 2022 to 2025), giving nine colour classes. Dark blue means high on both. A map shows where the two measures coincide, which a scatter cannot show. Tiers use average ranks, so ZIPs tied at zero share one tier.

**Why this chart answers the question:** TODO

**Finding:** 46 of the 226 ZIPs (20%) are high on both measures. They cluster in Brooklyn (24 of 39 ZIPs) and the Bronx (11 of 27). Queens has 10 of 67, Staten Island 1 of 12, and Manhattan none of its 81 ZIPs. Across all ZIPs, new-building filings and severe injuries are strongly rank-correlated (Spearman rho = 0.69).

**Result and interpretation:** TODO

**Caveats:**
- Association only, not causation, and busy areas have more of everything.
- Only 322 of 904 ZIP-years have a median build time, because only ZIPs with a signed-off new building have one. The count falls from 138 in 2022 to 28 in 2025 because recent buildings have not been signed off yet, so later years lean toward fast projects.

In [ ]:
# TODO (M5): Q9 chart

### Q10 (M5)

**Question:** Are ZIPs where new buildings stay under construction longer associated with more serious crashes?

**Planned chart:** zip by year; scatter of median days permit-to-signoff vs crashes, sized by new buildings

**Why this chart answers the question:** TODO

**Result and interpretation:** TODO

**Caveats:** TODO

In [ ]:
# TODO (M5): Q10 chart

### Q11 (M6)

**Question:** Do serious crashes in construction-heavy ZIPs occur at different hours than in quiet ZIPs?

**Planned chart:** row-level crashes tagged with ZIP tier; hour-by-weekday heatmaps faceted by tier

**Why this chart answers the question:** Q11 asks whether serious crashes in construction-heavy ZIPs occur at different times than in quiet ZIPs. Each ZIP is placed in a Low, Medium or High tier by its total new-building filings (ZIPs with the same total always share a tier), and each tier gets an hour-by-weekday heatmap. Each panel shows the share of that tier’s own crashes, so a tier with more crashes does not look hotter just for being bigger, and all three panels share one color scale. If the heat sits in different hours or days across tiers, for example rush hours versus nights and weekends, the chart answers the question.

**Result and interpretation:** The time-of-day pattern is similar in all three tiers: about a quarter of crashes fall in the afternoon peak (16:00 to 19:59), about 28% between 10:00 and 15:59, and about 13% in the morning peak (06:00 to 09:59). High-construction ZIPs have a slightly larger share of their crashes in the afternoon peak (26.1% against 24.6% in the Low tier) and the evening (19.0% against 17.4%), and a smaller share at night (14.2% against 16.1%). These differences are under 2 percentage points, do not form a steady gradient from Low to High (the Medium tier has the smallest afternoon share, 23.4%), and the overall test is borderline (chi-square p = 0.039, which is optimistic because crashes cluster within ZIPs). The weekend share is about the same in every tier (27.7%, 28.6% and 27.4%). Construction-heavy ZIPs do have many more serious crashes in total (about 72 per ZIP in the High tier against about 12 in the Low tier), but we find no strong evidence that these crashes happen at different hours.

**Caveats:** Q11 uses serious crashes from 2022 to 2025 (the complete years in the ZIP-year table) that have a ZIP and are not on a highway: 10,062 of the 13,309 crashes in the file, which also covers January to June 2026. Fatal crashes are much more likely to lack a ZIP (33.8% of fatal crashes versus 6.7% of non-fatal ones), so the picture leans toward non-fatal crashes. Highway crashes are excluded by default because almost none have an original ZIP. The Low tier has only 971 crashes (about 6 per hour-by-weekday cell, against about 22 for Medium and 32 for High), so its darkest cells are mostly noise, and they also stretch the shared color scale, which makes the other two panels look paler. Read broad patterns and the percentages, not single cells. Tiers are based on new-building filings, which are paperwork, not confirmed construction, and ZIPs with the same filing total always share a tier, so the tiers are not exactly equal in size (Low 79, Medium 72, High 75 ZIPs). This is an association, not causation.

In [ ]:
from pathlib import Path
import pandas as pd
from src.charts import q11

# The notebook may run from the repo root or from notebooks/, so find the repo root first.
ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
zip_year = pd.read_csv(ROOT / "data/clean/zip_year_integrated.csv", dtype={"zip": str})
crashes = pd.read_csv(ROOT / "data/clean/crash_clean.csv", dtype={"zip": str})

fig_q11 = q11.make_fig(q11.prepare(crashes, zip_year))
fig_q11.show()


### Q12 (M6)

**Question:** Can construction metrics, borough, and year identify ZIPs in the top quartile of serious crashes, and which feature matters most?

**Planned chart:** zip by year; random forest or logistic model with a feature-importance bar chart

**Why this chart answers the question:** Q12 asks whether construction activity carries
predictive signal about crash-heavy ZIPs, and how much compared with borough and year.
A classifier that predicts "top quartile of serious street crashes" from these features
turns that into a measurable quantity: held-out AUC. Permutation importance then ranks
the features by how much AUC drops when each one is shuffled, so the bar chart shows
which feature the model relies on most.

**Method.**
- Data: `zip_year_integrated.csv` (ZIP by year, 2022 to 2025).
- Target: ZIP-years in the top quartile of serious street crashes (`t_crashes_street`).
- Model: random forest (200 trees, `min_samples_leaf=3`, `class_weight="balanced"`,
  `random_state=42`).
- Validation: 5-fold `GroupKFold` grouped by ZIP, so the same ZIP never appears in both
  the training and test folds. Without grouping, a ZIP's rows in adjacent years would leak
  across the split and inflate performance.
- Importance: permutation importance on each held-out fold, scored by ROC AUC,
  `n_repeats=5`, averaged across the 5 folds.

**Design decisions.** `n_jobs=1` and `n_repeats=5` were chosen after profiling: the
original `n_jobs=-1` with `n_repeats=10` took about 94 s because of parallel-worker
overhead, and the final setting runs in about 3 s after the one-time imports. Reducing
repeats changes the importance estimates only slightly, since they are still averaged
over 5 folds.

**Limitations.** Importance shows what helps *predict* top-quartile ZIPs, not what
*causes* crashes. Permutation importance can split credit between correlated features
(for example, `h_new_building` and `h_proposed_units`). Crash coverage differs by year
(see D4 in the data contract), so `year` may partly reflect location coverage rather than
real change.

**Findings.** The 5-fold ZIP-grouped random forest achieved a mean ROC AUC of 0.899, with approximately 25.4% positive observations. The three most important features were total housing filings (0.0540), sidewalk-shed filings (0.0234), and proposed dwelling units (0.0139), measured by mean decrease in held-out AUC when permuted. Housing activity measures were more informative than year and individual borough indicators. These are predictive relationships, not causal effects.

**Result and interpretation:** On ZIPs it had not seen, the random forest separates top-quarter ZIP-years from the rest well (mean ROC AUC 0.90 across 5 folds, range 0.87 to 0.94; 0.50 is chance). Total filings is by far the most useful feature (shuffling it lowers the AUC by about 0.054), followed by sidewalk-shed filings (about 0.023), proposed dwelling units (about 0.014) and alteration CO filings (about 0.011). New-building filings add little (about 0.003), and demolition, curb-cut filings, year and most borough indicators are close to zero; among boroughs only the Bronx stands out (about 0.007). The signal therefore comes mainly from overall filing activity, which tracks how busy and built-up a ZIP is, and not from new construction specifically.

**Caveats:** Several housing features measure overlapping things (all filings include alterations, sidewalk sheds and new buildings), so permutation importance can understate each one individually: when one is shuffled, the model can recover the information from the others. The model has no population, road-length or traffic features, so total filings probably stands in for how busy a ZIP is. A missing median permit-to-sign-off time (64% of ZIP-years, mainly ZIPs with no new buildings) is filled with 0, so for that feature 0 mostly means "no data". Year has almost no importance by design, because the target is ranked within each year. The error bars show the spread across the 5 folds.

In [4]:

from pathlib import Path
import pandas as pd
import plotly.express as px
from src.charts import q12

# Find the repository root
ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent

# Load the same integrated data used by the website
zip_year = pd.read_csv(
    ROOT / "data/clean/zip_year_integrated.csv",
    dtype={"zip": str}
)

# Run the actual grouped cross-validation analysis
importance, auc, positive_share = q12.compute_importance(zip_year)

# Display verified numerical results
print("Q12 — Random Forest Feature Importance")
print(f"Mean ROC AUC: {auc:.3f}")
print(f"Positive class share: {positive_share:.3f}")

display(
    importance.sort_values("mean", ascending=False)
    .round(4)
)

# Plot actual permutation importance values
plot_data = importance.sort_values("mean", ascending=True)

fig_q12 = px.bar(
    plot_data,
    x="mean",
    y="label",
    error_x="std",
    orientation="h",
    title="Q12: Which Features Predict High-Crash ZIP-Years?",
    labels={
        "mean": "Mean decrease in held-out ROC AUC",
        "label": "Feature"
    },
    height=650
)

fig_q12.add_vline(x=0, line_dash="dash")
fig_q12.update_layout(yaxis_title="", xaxis_title="Permutation importance")
fig_q12.show()


Q12 — Random Forest Feature Importance
Mean ROC AUC: 0.899
Positive class share: 0.254


,feature,label,mean,std
0,h_filings_total,All filings,0.0540,0.0135
1,h_sidewalk_shed,Sidewalk-shed filings,0.0234,0.0103
2,h_proposed_units,Proposed dwelling units,0.0139,0.0080
3,h_alteration_co,Alteration CO filings,0.0108,0.0072
4,borough_Bronx,Borough: Bronx,0.0065,0.0034
5,borough_Queens,Borough: Queens,0.0041,0.0055
6,h_new_building,New-building filings,0.0033,0.0026
7,h_median_days_permit_to_signoff,Median days permit to sign-off,0.0019,0.0029
8,borough_Brooklyn,Borough: Brooklyn,0.0013,0.0037
9,borough_Manhattan,Borough: Manhattan,0.0012,0.0024


## 9. Story: findings, limitations, and impact

*Owner: M6.*

**Discovery.** What did we find? What surprised us? (TODO after results)

**Limits.**
- SIRTA covers only serious crashes analyzed by the DOT, not all crashes.
- Some crash locations had no ZIP and were recovered by spatial join; some remain unlocated.
- H5 filings are paperwork, not confirmed construction.
- Busy areas have more of everything, so counts can reflect population and traffic.
- All results are associations, not causation.

**Impact.** Who can use this, and what should they do differently? (TODO)

## 10. Prompts used

Every prompt used at any step, copied from `docs/prompt_log.md` at the end of the project.

## 11. Contributions

Each member's contributions (research questions, visualizations, cleaning, integration, website, other work), copied from `docs/contributions.md` at the end of the project.